In [ ]:
import os
import time
import numpy as np
import pandas as pd
import seaborn as sns
import ydata_profiling
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport

from ydata_profiling.model.summarizer import BaseSummarizer
from visions import VisionsTypeset
import json


In [2]:
df = pd.read_csv("Titanic_uncleaned.csv")

In [3]:
def generate_profile_data(df):
    """Generate a pandas-profiling report and parse it into a JSON-serializable dict."""
    profile = ProfileReport(
        df,
        samples=None,
        interactions=None,
        missing_diagrams=None,
    )
    json_str = profile.to_json()
    return json.loads(json_str)

In [4]:
def clean_profile_report(df):
    """Clean a generated profile report and return cleaned data structure."""
    data = generate_profile_data(df)

    #  top-level section
    for key in ["scatter", "missing", "package", "interactions"]:
        data.pop(key, None)

    # Table-level
    data["table"].pop("memory_size", None)
    data["table"].pop("record_size", None)

    COMMON_TEXT_METADATA = [
        "block_alias_values",
        "block_alias_counts",
        "n_block_alias",
        "block_alias_char_counts",
        "script_counts",
        "n_scripts",
        "script_char_counts",
        "category_alias_counts",
        "n_category",
        "category_alias_char_counts",
    ]

    CATEGORICAL_TEXT_METADATA = [
        "max_length",
        "mean_length",
        "median_length",
        "min_length",
        "n_characters_distinct",
        "n_characters",
        "character_counts",
        "word_counts",
        "category_alias_values",
    ]

    for var_name, var_data in data["variables"].items():
        var_data.pop("memory_size", None)
        var_data.pop("hashable", None)

        for key in ["histogram", "length_histogram", "histogram_length", "bin_edges"]:
            var_data.pop(key, None)

        col_type = var_data.get("type", "")
        value_counts = var_data.get("value_counts_index_sorted", {})

        if col_type == "Numeric":
            var_data["value_counts_index_sorted"] = dict(
                list(value_counts.items())[:10]
            )
        elif col_type == "Text":
            var_data.pop("value_counts_without_nan", None)
            var_data["value_counts_index_sorted"] = dict(
                list(value_counts.items())[:10]
            )
            for key in COMMON_TEXT_METADATA:
                var_data.pop(key, None)
            if "category_alias_values" in var_data:
                var_data["category_alias_values"] = dict(
                    list(var_data["category_alias_values"].items())[:10]
                )
            if "word_counts" in var_data:
                sorted_words = sorted(
                    var_data["word_counts"].items(), key=lambda x: x[1], reverse=True
                )[:10]
                var_data["word_counts"] = dict(sorted_words)
        elif col_type == "Categorical":
            for key in CATEGORICAL_TEXT_METADATA + COMMON_TEXT_METADATA:
                var_data.pop(key, None)

    return data


cleaned_data = clean_profile_report(df)

with open("titanic_profile_clean16.json", "w") as f:
    json.dump(cleaned_data, f, indent=2)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

/Users/noursameh/.pyenv/versions/3.11.6/lib/python3.11/site-packages/ydata_profiling/model/correlations.py:66: UserWarning: There was an attempt to calculate the auto correlation, but this failed.
To hide this warning, disable the calculation
(using `df.profile_report(correlations={"auto": {"calculate": False}})`
If this is problematic for your use case, please report this as an issue:
https://github.com/ydataai/ydata-profiling/issues
(include the error message: 'Function <code object pandas_auto_compute at 0x306a48a90, file "/Users/noursameh/.pyenv/versions/3.11.6/lib/python3.11/site-packages/ydata_profiling/model/pandas/correlations_pandas.py", line 164>')
  warnings.warn(


Render JSON:   0%|          | 0/1 [00:00<?, ?it/s]